# SSD_AIMO3 Thesis Validation Engine

This notebook is the primary engine for initial Colab experimentation.

Its job is not to produce optimistic demos. Its job is to help us **(in)validate the thesis** under a disciplined A0 -> A1 -> A5 ladder:

- `A0`: strongest fair frozen baseline
- `A1`: naive SSD self-distillation
- `A5`: tropical reranking over generated eval traces

Success means a decision-ready answer, not just runnable code.

## Zero-edit starter

This notebook defaults to a **self-contained starter run**.

If you open it in a fresh Colab and run top-to-bottom without editing anything, it will execute the replayable fixture ladder and emit a decision summary.

That starter result is a workflow proof, not a thesis result. When you are ready for an actual thesis test, switch `EXPERIMENT_MODE` from `"starter"` to `"real"` in the parameter cell.

## Method discipline

Before running anything, keep the evaluation contract fixed:

1. The primary metric is exact final-answer accuracy.
2. Baseline and student must use the same extraction rules.
3. A1 only counts as useful if it beats the best fair A0 baseline.
4. A5 only counts as useful if it improves over matched non-tropical evaluation on the same generated traces.
5. Any gain that comes from extraction bugs, prompt drift, or incomparable budgets is not a scientific win.

Use the final comparison outputs to answer:

- Did A1 beat A0?
- Did A5 add value beyond A1?
- Are the discordant-pair counts large enough to justify more budget?
- What failure modes dominated the losses?

In [ ]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path


PROJECT_ROOT = Path("/content/SSD_AIMO3")
RUN_NAME = "colab_notebook_session"
RUN_ROOT = PROJECT_ROOT / "runs" / RUN_NAME
BUNDLE_DIR = RUN_ROOT / "bundle"


def run(*args: str, cwd: Path | None = None) -> None:
    cmd = [str(arg) for arg in args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=str(cwd or PROJECT_ROOT), check=True)


def read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, data) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


def ensure_repo(repo_url: str, repo_ref: str) -> None:
    if (PROJECT_ROOT / ".git").exists():
        print(f"Using existing repo at {PROJECT_ROOT}")
        return
    if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f"{PROJECT_ROOT} exists but is not a git checkout. Remove it or point PROJECT_ROOT elsewhere.")
    run("git", "clone", "--depth", "1", "--branch", repo_ref, repo_url, str(PROJECT_ROOT), cwd=Path("/content"))
    print(f"Cloned {repo_url}@{repo_ref} -> {PROJECT_ROOT}")


PROJECT_ROOT

In [ ]:
# Default: fully self-contained starter run.
# Change EXPERIMENT_MODE to "real" when you want to run your own model and manifests.

EXPERIMENT_MODE = "starter"  # "starter" or "real"
REPO_URL = "https://github.com/fyremael/SSD_AIMO3.git"
REPO_REF = "main"

assert EXPERIMENT_MODE in {"starter", "real"}

if EXPERIMENT_MODE == "starter":
    RUN_NAME = "colab_starter_fixture"
    MODEL_ID = "fixture_replay_bundle"
    RAW_INPUT_PATH = ""
    RAW_INPUT_FORMAT = "auto"
    PROBLEM_ID_FIELD = "problem_id"
    PROMPT_FIELD = "prompt_text"
    ANSWER_FIELD = "gold_answer"
    TOPIC_FIELD = "topic"
    DIFFICULTY_FIELD = "difficulty"
    TAGS_FIELD = "tags"
    SOURCE_NAME = "fixture_starter"
    PROMPT_MANIFEST_JSONL = PROJECT_ROOT / "data" / "fixture_unlabeled_prompts.jsonl"
    EVAL_PROMPT_MANIFEST_JSONL = PROJECT_ROOT / "data" / "fixture_eval_samples_a0.jsonl"
    PROBLEM_METADATA_JSONL = PROJECT_ROOT / "data" / "fixture_problem_metadata.jsonl"
    NUM_EVAL_SAMPLES = 4
else:
    RUN_NAME = "colab_real_session"
    MODEL_ID = "REPLACE_WITH_HF_MODEL_ID"
    RAW_INPUT_PATH = ""  # optional: CSV / JSON / JSONL problem-level source
    RAW_INPUT_FORMAT = "auto"
    PROBLEM_ID_FIELD = "problem_id"
    PROMPT_FIELD = "prompt_text"
    ANSWER_FIELD = "gold_answer"
    TOPIC_FIELD = "topic"
    DIFFICULTY_FIELD = "difficulty"
    TAGS_FIELD = "tags"
    SOURCE_NAME = "colab_real_corpus"
    PROMPT_MANIFEST_JSONL = PROJECT_ROOT / "data" / "real_prompts.jsonl"
    EVAL_PROMPT_MANIFEST_JSONL = PROJECT_ROOT / "data" / "real_eval_prompts.jsonl"
    PROBLEM_METADATA_JSONL = PROJECT_ROOT / "data" / "real_problem_metadata.jsonl"
    NUM_EVAL_SAMPLES = 8

RUN_ROOT = PROJECT_ROOT / "runs" / RUN_NAME
BUNDLE_DIR = RUN_ROOT / "bundle"
LADDER_ROOT = RUN_ROOT / "starter_fixture_ladder"

print(RUN_ROOT)

In [ ]:
# If this Colab session does not already have the repo checked out, clone it now.

ensure_repo(REPO_URL, REPO_REF)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)

In [ ]:
# In Colab, run this once per fresh session after the repo is present.
# If deps are already installed, you can skip or comment out the pip line.

run("python", "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements" / "colab-gpu.txt"))
run("python", str(PROJECT_ROOT / "scripts" / "colab_runtime_probe.py"), "--output-json", str(RUN_ROOT / "colab_runtime.json"))
runtime = read_json(RUN_ROOT / "colab_runtime.json")
print(json.dumps(runtime, indent=2))

In [ ]:
# If RAW_INPUT_PATH is set in real mode, normalize the source into the repo's prompt/eval/metadata contracts.

if EXPERIMENT_MODE == "starter":
    print("Starter mode uses the built-in fixture manifests.")
elif RAW_INPUT_PATH:
    run(
        "python",
        str(PROJECT_ROOT / "scripts" / "build_problem_manifests.py"),
        "--input-path",
        RAW_INPUT_PATH,
        "--input-format",
        RAW_INPUT_FORMAT,
        "--prompt-output-jsonl",
        str(PROMPT_MANIFEST_JSONL),
        "--eval-output-jsonl",
        str(EVAL_PROMPT_MANIFEST_JSONL),
        "--metadata-output-jsonl",
        str(PROBLEM_METADATA_JSONL),
        "--summary-json",
        str(RUN_ROOT / "manifest_summary.json"),
        "--problem-id-field",
        PROBLEM_ID_FIELD,
        "--prompt-field",
        PROMPT_FIELD,
        "--answer-field",
        ANSWER_FIELD,
        "--topic-field",
        TOPIC_FIELD,
        "--difficulty-field",
        DIFFICULTY_FIELD,
        "--tags-field",
        TAGS_FIELD,
        "--source-name",
        SOURCE_NAME,
    )

print(PROMPT_MANIFEST_JSONL)
print(EVAL_PROMPT_MANIFEST_JSONL)
print(PROBLEM_METADATA_JSONL)

In [ ]:
# Materialize a clean Colab config bundle from one parameter surface for real runs.

if EXPERIMENT_MODE == "starter":
    print("Starter mode uses the fixture configs directly and skips bundle materialization.")
else:
    run(
        "python",
        str(PROJECT_ROOT / "scripts" / "materialize_colab_bundle.py"),
        "--output-dir",
        str(BUNDLE_DIR),
        "--model-id",
        MODEL_ID,
        "--prompt-manifest-jsonl",
        str(PROMPT_MANIFEST_JSONL),
        "--eval-prompt-manifest-jsonl",
        str(EVAL_PROMPT_MANIFEST_JSONL),
        "--problem-metadata-jsonl",
        str(PROBLEM_METADATA_JSONL),
    )
    bundle_manifest = read_json(BUNDLE_DIR / "bundle_manifest.json")
    print(json.dumps(bundle_manifest, indent=2))

## Run A0 and A1

The early goal is not leaderboard chasing. The goal is to establish whether the student has any directional lift over the strongest fair frozen baseline under the same evaluation contract.

In [ ]:
# A0 baseline generation + evaluation

if EXPERIMENT_MODE == "starter":
    run(
        "python",
        str(PROJECT_ROOT / "scripts" / "run_validation_ladder.py"),
        "--output-dir",
        str(LADDER_ROOT),
    )
    print(read_json(LADDER_ROOT / "a0_eval" / "metrics.json"))
    print(read_json(LADDER_ROOT / "a1_train" / "training_plan.json"))
else:
    run(
        "python", str(PROJECT_ROOT / "scripts" / "generate_self_samples.py"),
        "--config", str(BUNDLE_DIR / "a0.yaml"),
        "--input-jsonl", str(EVAL_PROMPT_MANIFEST_JSONL),
        "--num-samples", str(NUM_EVAL_SAMPLES),
        "--output-dir", str(RUN_ROOT / "a0_eval_generation"),
    )
    run(
        "python", str(PROJECT_ROOT / "scripts" / "run_eval_math.py"),
        "--config", str(BUNDLE_DIR / "a0.yaml"),
        "--input-jsonl", str(RUN_ROOT / "a0_eval_generation" / "generations.jsonl"),
        "--output-dir", str(RUN_ROOT / "a0_eval"),
    )

    # A1 self-sample generation + training launch
    run(
        "python", str(PROJECT_ROOT / "scripts" / "generate_self_samples.py"),
        "--config", str(BUNDLE_DIR / "a1.yaml"),
        "--input-jsonl", str(PROMPT_MANIFEST_JSONL),
        "--output-dir", str(RUN_ROOT / "a1_self_samples"),
    )
    run(
        "python", str(PROJECT_ROOT / "scripts" / "train_ssd_math.py"),
        "--config", str(BUNDLE_DIR / "a1.yaml"),
        "--input-jsonl", str(RUN_ROOT / "a1_self_samples" / "generations.jsonl"),
        "--output-dir", str(RUN_ROOT / "a1_train"),
        "--launch",
    )

    print(read_json(RUN_ROOT / "a0_eval" / "metrics.json"))
    print(read_json(RUN_ROOT / "a1_train" / "training_plan.json"))

In [ ]:
# Re-materialize the bundle with the trained adapter path so the student can be evaluated honestly.

if EXPERIMENT_MODE == "starter":
    print(read_json(LADDER_ROOT / "a1_eval" / "metrics.json"))
else:
    ADAPTER_PATH = RUN_ROOT / "a1_train" / "adapter"
    assert ADAPTER_PATH.exists(), f"Expected adapter at {ADAPTER_PATH}"

    run(
        "python",
        str(PROJECT_ROOT / "scripts" / "materialize_colab_bundle.py"),
        "--output-dir",
        str(BUNDLE_DIR),
        "--model-id",
        MODEL_ID,
        "--prompt-manifest-jsonl",
        str(PROMPT_MANIFEST_JSONL),
        "--eval-prompt-manifest-jsonl",
        str(EVAL_PROMPT_MANIFEST_JSONL),
        "--problem-metadata-jsonl",
        str(PROBLEM_METADATA_JSONL),
        "--student-adapter-path",
        str(ADAPTER_PATH),
    )

    run(
        "python", str(PROJECT_ROOT / "scripts" / "generate_self_samples.py"),
        "--config", str(BUNDLE_DIR / "a1_student_eval.yaml"),
        "--input-jsonl", str(EVAL_PROMPT_MANIFEST_JSONL),
        "--num-samples", str(NUM_EVAL_SAMPLES),
        "--output-dir", str(RUN_ROOT / "a1_eval_generation"),
    )
    run(
        "python", str(PROJECT_ROOT / "scripts" / "run_eval_math.py"),
        "--config", str(BUNDLE_DIR / "a1_student_eval.yaml"),
        "--input-jsonl", str(RUN_ROOT / "a1_eval_generation" / "generations.jsonl"),
        "--output-dir", str(RUN_ROOT / "a1_eval"),
    )

    print(read_json(RUN_ROOT / "a1_eval" / "metrics.json"))

In [ ]:
# Run A5 on the exact same A1 eval generations, then compare A0 vs A1 and A1 vs A5.

if EXPERIMENT_MODE == "starter":
    a0_metrics = read_json(LADDER_ROOT / "a0_eval" / "metrics.json")
    a1_metrics = read_json(LADDER_ROOT / "a1_eval" / "metrics.json")
    a5_metrics = read_json(LADDER_ROOT / "a5_eval" / "metrics.json")
    a0_vs_a1 = read_json(LADDER_ROOT / "compare_a0_a1" / "comparison_summary.json")
    a1_vs_a5 = read_json(LADDER_ROOT / "compare_a1_a5" / "comparison_summary.json")
else:
    run(
        "python", str(PROJECT_ROOT / "scripts" / "run_eval_math.py"),
        "--config", str(BUNDLE_DIR / "a5.yaml"),
        "--input-jsonl", str(RUN_ROOT / "a1_eval_generation" / "generations.jsonl"),
        "--output-dir", str(RUN_ROOT / "a5_eval"),
    )
    run(
        "python", str(PROJECT_ROOT / "scripts" / "compare_eval_runs.py"),
        "--run-a-dir", str(RUN_ROOT / "a0_eval"),
        "--run-b-dir", str(RUN_ROOT / "a1_eval"),
        "--run-a-label", "a0_majority_vote",
        "--run-b-label", "a1_majority_vote",
        "--metadata-jsonl", str(PROBLEM_METADATA_JSONL),
        "--output-dir", str(RUN_ROOT / "compare_a0_a1"),
    )
    run(
        "python", str(PROJECT_ROOT / "scripts" / "compare_eval_runs.py"),
        "--run-a-dir", str(RUN_ROOT / "a1_eval"),
        "--run-b-dir", str(RUN_ROOT / "a5_eval"),
        "--run-a-label", "a1_majority_vote",
        "--run-b-label", "a5_tropical_rerank",
        "--metadata-jsonl", str(PROBLEM_METADATA_JSONL),
        "--output-dir", str(RUN_ROOT / "compare_a1_a5"),
    )

    a0_metrics = read_json(RUN_ROOT / "a0_eval" / "metrics.json")
    a1_metrics = read_json(RUN_ROOT / "a1_eval" / "metrics.json")
    a5_metrics = read_json(RUN_ROOT / "a5_eval" / "metrics.json")
    a0_vs_a1 = read_json(RUN_ROOT / "compare_a0_a1" / "comparison_summary.json")
    a1_vs_a5 = read_json(RUN_ROOT / "compare_a1_a5" / "comparison_summary.json")

def accuracy_metric(metrics):
    return metrics.get("exact_final_answer_accuracy", metrics.get("accuracy"))

a0_accuracy = accuracy_metric(a0_metrics)
a1_accuracy = accuracy_metric(a1_metrics)
a5_accuracy = accuracy_metric(a5_metrics)

decision_summary = {
    "run_root": str(RUN_ROOT),
    "experiment_mode": EXPERIMENT_MODE,
    "model_id": MODEL_ID,
    "a0_accuracy": a0_accuracy,
    "a1_accuracy": a1_accuracy,
    "a5_accuracy": a5_accuracy,
    "a1_minus_a0_accuracy": (a1_accuracy or 0.0) - (a0_accuracy or 0.0),
    "a5_minus_a1_accuracy": (a5_accuracy or 0.0) - (a1_accuracy or 0.0),
    "a0_vs_a1_discordant_pairs": a0_vs_a1.get("discordant_pairs"),
    "a0_vs_a1_sign_test_pvalue": a0_vs_a1.get("paired_sign_test_pvalue_two_sided"),
    "a1_vs_a5_discordant_pairs": a1_vs_a5.get("discordant_pairs"),
    "a1_vs_a5_sign_test_pvalue": a1_vs_a5.get("paired_sign_test_pvalue_two_sided"),
    "thesis_status": "starter_fixture_complete" if EXPERIMENT_MODE == "starter" else "undecided_review_outputs",
}
write_json(RUN_ROOT / "decision_summary.json", decision_summary)

print("A0 metrics:\n", json.dumps(a0_metrics, indent=2))
print("A1 metrics:\n", json.dumps(a1_metrics, indent=2))
print("A5 metrics:\n", json.dumps(a5_metrics, indent=2))
print("A0 vs A1:\n", json.dumps(a0_vs_a1, indent=2))
print("A1 vs A5:\n", json.dumps(a1_vs_a5, indent=2))
print("Decision summary:\n", json.dumps(decision_summary, indent=2))

## Decision pass

Use the outputs above to write down an explicit conclusion before changing prompts or budgets again:

- `Proceed`: A1 beats A0 under a fair contract and the gain looks robust enough to merit more budget.
- `Proceed conditionally`: A1 is weak but A5 or a later controlled extension adds value.
- `Do not proceed`: gains vanish, depend on bad extraction, or fail under fair paired comparison.

Red-team questions:

- Did we accidentally give the student a different inference budget?
- Did extraction success change enough to explain the gain?
- Are the discordant-pair counts large enough to trust the direction?
- Which topic slices improved, and which did not?
- What is the cheapest next experiment that would most strongly falsify the current interpretation?